# 08 - AQI Category Classification and Model Evaluation
**Project:** Air Quality & Pollution Intelligence - Data Mining and Business Intelligence

**Objective:** Predict the AQI band from pollutant concentrations with several algorithms, avoid data leakage, respect the class imbalance, and compare the models on a held-out split.

**How to read this notebook:** every number printed below is produced by the
code in this notebook from `data/raw/Air_quality_data.csv`. Column names are
discovered at runtime through `src/config.py`, so nothing is assumed.


In [ ]:
"""Environment bootstrap: make src/ importable and pin the working directory."""
import sys, os, warnings
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)
%matplotlib inline
print("project root:", PROJECT_ROOT)

In [ ]:
import config as C
import data_utils as U
import preprocessing as P
import feature_engineering as FE
import classification as CL
import figures as F
import viz as V
from IPython.display import display, Image

clean, *_ = P.clean(U.load_raw())
feat, _ = FE.build_features(clean)
polls = C.pollutant_columns(feat.columns)
target = C.TARGET_COL
model_df = CL.drop_non_informative(feat, target)
print("records with a usable AQI band:", f"{len(model_df):,}")

## 1. Target and class balance

Accuracy is meaningless without the class distribution, so the balance is measured
first and a prior-based baseline is carried through every comparison.

In [ ]:
balance = CL.class_balance(model_df, target)
display(balance)
print(f"imbalance ratio (largest/smallest class): {balance.attrs['imbalance_ratio']}")
print(f"majority class: {balance.attrs['majority_class']} "
      f"({balance.attrs['majority_share_pct']}%) -> baseline accuracy")

## 2. Data leakage: the AQI column must not be a predictor

`AQI_Bucket` is a banding of `AQI` (proved in notebook 02), so giving a model the
AQI number to predict the AQI band is predicting the target from itself. The cell
below measures how much that mistake would inflate the score.

In [ ]:
leak = CL.leakage_check(model_df, polls, target)
display(pd.DataFrame([leak]).T.rename(columns={0: "value"})
        if False else pd.Series(
            {"accuracy without AQI (%)": leak["without leaky column"]["accuracy_%"],
             "accuracy with AQI (%)": leak["with leaky column"]["accuracy_%"],
             "inflation (pp)": leak["accuracy_inflation_pp"],
             "macro F1 without AQI": leak["without leaky column"]["macro_F1"],
             "macro F1 with AQI": leak["with leaky column"]["macro_F1"]}).to_frame("value"))
print()
print(leak["verdict"])
(C.PROCESSED_DIR / "leakage_check.json").write_text(__import__("json").dumps(leak, indent=2),
                                                    encoding="utf-8")

## 3. Models and their settings

| Model | Why included | Key settings |
|---|---|---|
| Baseline (prior) | reference point that any real model must beat | always predicts the largest class |
| Logistic Regression | linear, interpretable multi-class reference | L2, one-vs-rest internals, `class_weight="balanced"`, features scaled |
| Decision Tree | single transparent model, non-linear | `max_depth=12`, `min_samples_leaf=20`, balanced weights |
| Random Forest | bagged ensemble, robust, gives feature importances | 200 trees, `min_samples_leaf=2`, `balanced_subsample` |
| Hist Gradient Boosting | modern boosted benchmark | 250 iterations, learning rate 0.1 |

All use `random_state=42`, a stratified 75/25 split, and 3-fold stratified CV for
the accuracy column.

In [ ]:
comp, reports, matrices, split_info, pred_frame, y_test, X_test = CL.train_and_evaluate(
    model_df, polls, target, cv_folds=3)
display(comp)
print()
print("features given to the models:", split_info["features_used"])
print("excluded as leaky          :", split_info["excluded_leaky_column"])
print("train / test rows          :", split_info["train_rows"], "/", split_info["test_rows"])

In [ ]:
comp.to_csv(C.MODEL_COMPARISON_CSV, index=False)
display(comp[["Model", "Accuracy_%", "Precision_macro", "Recall_macro", "F1_macro",
             "Lift_over_Baseline_pp", "CV_Accuracy_%"]])

## 4. Why macro-F1 and per-class recall, not just accuracy

The four main bands hold over 99% of records; `Good` holds 6 rows and
`Satisfactory` 113. A model can therefore look excellent overall while never
detecting the rare classes.

In [ ]:
per_class = pd.concat([CL.report_to_frame(n, r) for n, r in reports.items()],
                     ignore_index=True)
rare = per_class[per_class["Class"].isin(["Good", "Satisfactory"])]
display(rare)
print("Support is the number of test records in that class; recall is the share found.")

In [ ]:
best = comp.iloc[0]["Model"]
display(pd.DataFrame(reports[best]).T.round(3))

## 5. Confusion matrices

In [ ]:
conf = pd.concat([CL.confusion_to_frame(m, sorted(split_info["classes"]), n)
                  for n, m in matrices.items()], ignore_index=True)
conf.to_csv(C.PROCESSED_DIR / "confusion_matrices.csv", index=False)
per_class.to_csv(C.PROCESSED_DIR / "classification_per_class.csv", index=False)
Image(F.classification(comp, conf, per_class, pd.DataFrame(),
                       sorted(split_info["classes"]))["02_confusion_matrices"])

In [ ]:
imp_model, imp_features = CL.fit_reference_model(model_df, polls, target)
imp = CL.feature_importance(imp_model, imp_features)
paths = F.classification(comp, conf, per_class, imp,
                         sorted(split_info["classes"]))
Image(paths["01_model_comparison"])

In [ ]:
Image(paths["03_per_class_recall"])
Image(paths["04_feature_importance"])

## 6. Which pollutants carry the signal?

Mean-decrease-in-impurity importance from a random forest trained on the full
frame. This is a *model-internal* measure: it says which columns the trees split
on, not which pollutants cause poor air quality.

In [ ]:
display(imp)
imp.to_csv(C.PROCESSED_DIR / "feature_importance.csv", index=False)

## 7. Test-set predictions exported for the dashboard

Actual-versus-predicted records allow the BI layer to show model reliability
without retraining anything in Power BI.

In [ ]:
pred_frame.to_csv(C.CLASSIFICATION_CSV, index=False)
print("saved:", C.CLASSIFICATION_CSV.name, "rows:", f"{len(pred_frame):,}")
display(pred_frame.head(8))

## Verification checklist

- [x] Leak-proof feature set (AQI excluded, and the inflation measured)
- [x] Stratified split, fixed random state
- [x] Baseline model reported next to every real model
- [x] Class imbalance quantified before accuracy is interpreted
- [x] Precision, recall and F1 reported per class and macro-averaged
- [x] Confusion matrices and feature importances saved

**Common errors**

| Error | Meaning | Fix |
|---|---|---|
| `ConvergenceWarning` in logistic regression | unscaled inputs or too few iterations | features are piped through `StandardScaler` and `max_iter=2000` |
| `UndefinedMetricWarning` | a class has no predicted samples | `zero_division=0` is set; recall is reported instead |
| Accuracy of ~100% | leakage | re-check `leakage_check` above |

**Next:** `09_final_analysis.ipynb`.